# 🩺 Fine-tuning LoRA — Modèle médical expérimental

**Projet :** TechCorp Challenge IA — Mission Bonus R&D

**Objectif :** fine-tuner un modèle de base léger (TinyLlama 1.1B) avec LoRA sur un dataset de conversations médicales, pour produire un PoC interne. **Ce modèle reste expérimental** et ne sera pas déployé en production.

## Stack
- Modèle de base : `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (rapide, tient en mémoire Colab T4)
- Dataset : `ruslanmv/ai-medical-chatbot` (Hugging Face)
- Méthode : LoRA via `peft` + quantization 4-bit `bitsandbytes`
- Framework d'entraînement : `transformers.Trainer`

## Pré-requis Colab
Runtime → Type d'exécution → **GPU T4** (gratuit suffit pour la démo).

## 1. Installation

In [ ]:
!pip -q install \
    "torch>=2.1.0" \
    "transformers>=4.45.0" \
    "peft>=0.12.0" \
    "accelerate>=0.34.0" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.20.0" \
    "trl>=0.10.0"

In [ ]:
import torch
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

## 2. Chargement du dataset médical

On limite volontairement à **2 000 exemples** pour rester dans les contraintes de temps (GPU Colab gratuit).

In [ ]:
from datasets import load_dataset

raw = load_dataset('ruslanmv/ai-medical-chatbot', split='train[:2000]')
print('Colonnes :', raw.column_names)
print('Exemple :', raw[0])

In [ ]:
# Le dataset a typiquement les colonnes : Description, Patient, Doctor
# On formate au format chat de TinyLlama (template ChatML simplifié).

SYSTEM_PROMPT = (
    "You are a helpful medical assistant. "
    "You provide educational information only and always recommend consulting a licensed physician."
)

def format_example(ex):
    user = (ex.get('Patient') or ex.get('patient') or '').strip()
    doctor = (ex.get('Doctor') or ex.get('doctor') or '').strip()
    if not user or not doctor:
        return {'text': None}
    text = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{user}</s>\n"
        f"<|assistant|>\n{doctor}</s>"
    )
    return {'text': text}

formatted = raw.map(format_example, remove_columns=raw.column_names)
formatted = formatted.filter(lambda r: r['text'] is not None)
print('Après formatage :', len(formatted), 'exemples')
print(formatted[0]['text'][:500])

## 3. Modèle de base + quantization 4-bit

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Tokenisation

In [ ]:
MAX_LEN = 512

def tokenize(batch):
    enc = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
    )
    enc['labels'] = enc['input_ids'].copy()
    return enc

tokenized = formatted.map(tokenize, batched=True, remove_columns=['text'])
split = tokenized.train_test_split(test_size=0.05, seed=42)
print(split)

## 5. Entraînement

1 epoch suffit pour le PoC (≈15 min sur T4).

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='./medical_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to='none',
    remove_unused_columns=False,
)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=collator,
)

trainer.train()

In [ ]:
trainer.save_model('./medical_lora_final')
tokenizer.save_pretrained('./medical_lora_final')
print('✅ Adapter sauvegardé dans ./medical_lora_final')

## 6. Test d'inférence

In [ ]:
model.eval()

def ask(question, max_new_tokens=180):
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{question}</s>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.4,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return decoded.strip()

for q in [
    'I have had a persistent headache for 3 days. What could it be?',
    'What are the typical symptoms of type 2 diabetes?',
    'Can you explain what hypertension is?',
]:
    print('\n👤', q)
    print('🤖', ask(q))
    print('-' * 60)

## 7. (Optionnel) Téléchargement de l'adapter

L'adapter LoRA pèse ~10 Mo, on peut le récupérer pour démo.

```python
from google.colab import files
import shutil
shutil.make_archive('medical_lora_final', 'zip', './medical_lora_final')
files.download('medical_lora_final.zip')
```

## ✅ Résumé
- Base model : TinyLlama 1.1B (4-bit)
- LoRA : r=8, α=16, ~3 M params trainables
- Dataset : 2 000 conversations médicales (PoC)
- Temps d'entraînement : ~15 min sur Colab T4
- **À ne pas déployer en production** — PoC R&D uniquement.